# TOPOTEX Technical Report

当前主线：**Orientation-Aligned Single-Image Prototype**。

## Orientation-Aligned Single-Image Prototype

```
Canonical Mesh          One Stochastic Rendered Image
      |                        |
  Face Tokenizer        SingleImageEncoder
      |                        |
      +--- Face-Image Cross Attention ---+
                       |
                      Z_F  [F, 384]
                       |
              Factorized UV Query
                       |
                Flow Matching -> Texture
```

- 数据：Objaverse-OA canonical mesh（对齐已烘焙，`transform.json` 记录
  R/t/s 与来源）+ 每对象 6 张随机相机渲染（az U[0,360)、el U[-10,50],
  相机参数只是记录用元数据，**不输入模型**）。
- 训练：每步随机抽 1 张存储视图作条件（eval 固定 view_000）;
  native / xatlas / connected-partial 三查询采样。partial 是表面子集
  查询，不是 unwrap family。
- 冻结复用：FaceTokenizer、Topology Transformer、Z_F、Factorized UV
  Query、Global UV Query Attention、Flow Matching——与观测分支解耦。
- 验证工件：`topotex_OA_study/results/oa_prototype_report.md`（五问一答
  + GO/STOP）、`experiments/oa_prototype/record.json`（provenance）。

In [ ]:
# one real forward through the prototype — every stage shape
import json, sys
from pathlib import Path
p = Path.cwd()
PROJECT_ROOT = next(c for c in (p, *p.parents) if (c / "configs").exists())
sys.path.insert(0, str(PROJECT_ROOT))
import torch
from topotex import TopoTexDataset, TopoTexPipeline

OA_ROOT = Path("/root/youjiaZhang/topotex_data_OA")
cks = sorted((OA_ROOT / "runs").glob("oa80_*/ckpt.pt"))
ck = max(cks, key=lambda q: q.stat().st_mtime)
pipe = TopoTexPipeline.from_checkpoint(ck, "cuda:0")
model = pipe.model
assert model.conditioner.image_encoder_kind == "single"
man = [json.loads(l) for l in open(OA_ROOT / "dataset/manifest.jsonl")]
it = TopoTexDataset(str(OA_ROOT / "dataset"), [man[0]["sample_id"]], device="cuda:0")[0]
imgs = (it["mv_images"].float() / 255)[None]
with torch.no_grad():
    tok = model.conditioner.image_encoder(imgs[:, 0])
    z, _ = model.conditioner.encode_faces(it["mesh"], imgs, it["graph"])
    q = it["uv_queries"][0]
    o = model.condition(z, q)
    x = model.generate(o["uv_condition"], q["valid_mask"], num_steps=50, seed=20260727)
print(f"checkpoint                  {ck.parent.name} @ step {pipe.checkpoint.get('global_step')}")
print(f"stochastic input image      {tuple(imgs[:, 0].shape)}")
print(f"image tokens                {tuple(tok.shape)}")
print(f"face count                  {len(it['mesh']['faces'])}")
print(f"Z_F                         {tuple(z.shape)}")
print(f"uv_condition                {tuple(o['uv_condition'].shape)}")
print(f"texture                     {tuple(x.shape)}")

## Previous baseline（简注，不再是主流程）

此前的正式基线以六张离线生成的视图作观测（多视图 encoder + 同一套
Z_F/UV Query/FM 下游），在 4.6K 对象上验证了对象级泛化协议
（split by object / augment by UV query / evaluate on unseen objects）
与 dim384 配方的训练性质。其代码路径（`MultiViewEncoder`）与数据资产
完整保留；单图原型若达成 GO 决策，将在下一阶段的数据规模上与之正式
对比。细节见 git history 与 `experiments/experiment_log.md`。